In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import os
import re
import shutil
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from yolo_tools import get_yolo_label_df

In [20]:
root_dir = r'/localnvme/data/billboard/fused_data/data7436_mseg_c6_0917'
image_dir = os.path.join(root_dir, 'images')
label_dir = os.path.join(root_dir, 'labels')
output_dir = r'/localnvme/data/billboard/fused_data/data4197_mseg_c6_0917'
# output_dir = r'/localnvme/data/billboard/fused_data/data3617_mseg_c6_0917'
output_image_dir = os.path.join(output_dir, 'images')
output_label_dir = os.path.join(output_dir, 'labels')
defect_list = ['deformation', 'broken', 'abandonment', 'corrosion']
bd_dir = r'/localnvme/data/billboard/bd_data/data687_mseg_c6_0917'

In [13]:
def find_matching_files(folder_path):
    pattern1 = r"^cam_.*_cam_image_(20250722|20250812).*$"  # cam_..._cam_image_20250722...
    pattern2 = r"^camera\d_.*_(20250722|20250812).*$"      # cameraX_..._20250812...
    pattern3 = r".*_(20250722|20250812).*$"      # cameraX_..._20250812...

    matching_files = []
    for filename in os.listdir(folder_path):
        if re.match(pattern1, filename) or re.match(pattern2, filename) or re.match(pattern3, filename):
            matching_files.append(filename)
    matching_files.sort()
    return matching_files

In [14]:
ymt_data_list = find_matching_files(image_dir)

In [15]:
def get_stem2img_dict(img_dir):
    img_list = [img_name for img_name in os.listdir(img_dir)]
    stem_list = [Path(img).stem for img in img_list]
    stem2img_dict = dict(zip(stem_list, img_list))
    return stem2img_dict

In [16]:
def find_defect(label_dir, image_dir, defect_list, exclude_dir=None):
    defect_file_list = []
    exclude_list = os.listdir(exclude_dir) if exclude_dir else None
    label_file_list = os.listdir(label_dir)
    stem2img_dict = get_stem2img_dict(image_dir)
    for label_name in tqdm(label_file_list):
        input_label_path = os.path.join(label_dir, label_name)
        df = get_yolo_label_df(input_label_path, mdet=True, attributes=defect_list)
        with_defect = (df[defect_list] > 0).any().any()
        if with_defect:
            image_name = stem2img_dict[Path(label_name).stem]
            if exclude_list is not None and image_name in exclude_list:
                continue
            defect_file_list.append(image_name)
    return defect_file_list

In [17]:
defect_file_list = find_defect(label_dir, image_dir, defect_list, bd_dir)

100%|██████████| 7347/7347 [00:41<00:00, 176.60it/s]


In [18]:
select_list = list(set(ymt_data_list + defect_file_list))

In [19]:
print(f'find {len(select_list)} file, with {len(defect_file_list)} defects and {len(ymt_data_list)} ymt')

find 4064 file, with 2001 defects and 2288 ymt


In [21]:
os.makedirs(output_image_dir, exist_ok=True)
os.makedirs(output_label_dir, exist_ok=True)
for image_name in tqdm(select_list):
    label_name = Path(image_name).with_suffix('.txt')
    input_image_path = os.path.join(image_dir, image_name)
    output_image_path = os.path.join(output_image_dir, image_name)
    input_label_path = os.path.join(label_dir, label_name)
    output_label_path = os.path.join(output_label_dir, label_name)
    shutil.copy(input_image_path, output_image_path)
    shutil.copy(input_label_path, output_label_path)

100%|██████████| 4064/4064 [00:04<00:00, 897.09it/s] 


In [47]:
def random_select_exclude(data_dir, exclude_image_list, save_dir=None, train_ratio=0.9, random_seed=1010, full_path=True, suffix=''):
    image_dir = os.path.join(data_dir, 'images')
    label_dir = os.path.join(data_dir, 'labels')
    file_list = os.listdir(image_dir)
    if label_dir is not None:
        label_list = os.listdir(label_dir)
        label_list = [Path(label_name).stem for label_name in label_list]
        file_list_check = []
        for img_name in tqdm(file_list, desc='img check', total=len(file_list)):
            name = Path(img_name).stem
            if name in label_list:
                file_list_check.append(img_name)
        file_list = file_list_check
    if save_dir is None:
        save_dir = os.path.dirname(image_dir)
    src_num = len(file_list)
    file_list = [filename for filename in file_list if filename not in exclude_image_list]
    dst_num = len(file_list)
    print(f'{src_num} --> {dst_num}, exclude {src_num-dst_num} in {len(file_list)}')
    if full_path:
        file_list = [os.path.join(image_dir, filename) for filename in file_list]
    np.random.seed(random_seed)
    np.random.shuffle(file_list)
    train_num = int(len(file_list)*train_ratio)


    train_list = file_list[:train_num]
    val_list = file_list[train_num:]

    df_train = pd.DataFrame({'filename': train_list})
    df_val = pd.DataFrame({'filename': val_list})
    df_all = pd.DataFrame({'filename': train_list+val_list})
    df_train.to_csv(os.path.join(save_dir, f'train{suffix}.txt'), header=None, index=None)
    df_val.to_csv(os.path.join(save_dir, f'val{suffix}.txt'), header=None, index=None)
    df_all.to_csv(os.path.join(save_dir, 'all.txt'), header=None, index=None)
    print('%d save to %s,\n%d save to %s!'%(len(train_list), os.path.join(save_dir, f'train{suffix}.txt'),
                                           len(val_list), os.path.join(save_dir, f'val{suffix}.txt')))

In [48]:
random_select_exclude(output_dir, ymt_data_list, save_dir=None, train_ratio=0.8, random_seed=1010, full_path=True, suffix='_80p')
random_select_exclude(output_dir, ymt_data_list, save_dir=None, train_ratio=0.75, random_seed=1010, full_path=True, suffix='_75p')
random_select_exclude(output_dir, ymt_data_list, save_dir=None, train_ratio=0.70, random_seed=1010, full_path=True, suffix='_70p')
random_select_exclude(output_dir, ymt_data_list, save_dir=None, train_ratio=0.65, random_seed=1010, full_path=True, suffix='_65p')
random_select_exclude(output_dir, ymt_data_list, save_dir=None, train_ratio=0.60, random_seed=1010, full_path=True, suffix='_60p')

img check: 100%|██████████| 3617/3617 [00:00<00:00, 37848.74it/s]


3617 --> 1329, exclude 2288 in 1329
1063 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/train_80p.txt,
266 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/val_80p.txt!


img check: 100%|██████████| 3617/3617 [00:00<00:00, 45510.94it/s]


3617 --> 1329, exclude 2288 in 1329
996 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/train_75p.txt,
333 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/val_75p.txt!


img check: 100%|██████████| 3617/3617 [00:00<00:00, 40848.15it/s]


3617 --> 1329, exclude 2288 in 1329
930 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/train_70p.txt,
399 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/val_70p.txt!


img check: 100%|██████████| 3617/3617 [00:00<00:00, 30779.34it/s]


3617 --> 1329, exclude 2288 in 1329
863 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/train_65p.txt,
466 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/val_65p.txt!


img check: 100%|██████████| 3617/3617 [00:00<00:00, 45441.96it/s]


3617 --> 1329, exclude 2288 in 1329
797 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/train_60p.txt,
532 save to /localnvme/data/billboard/fused_data/data3617_mseg_c6_0915/val_60p.txt!
